# GRPO（LoRA）で 4B を学習する（Colab A100）

前提: Drive に `pt_frames_f32_s640.zip`（Perception Test のフレーム）がある。
学習セットは `data/perception_test/train_grpo.jsonl`（2,740 問）、評価は `dev_grpo.jsonl`（300 問）。

流れ: 学習前の dev 正解率 → GRPO（チェックポイントは Drive に 20 ステップごと）→ 学習後の dev 正解率。
まず `--max-steps 20` で 1 ステップの時間とメモリを測り、問題なければ 300 ステップに伸ばす。途中で切れても同じセルの再実行で最新のチェックポイントから続く。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. リポジトリと依存パッケージ

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
!pip install -q "transformers>=5.0" "accelerate>=1.0" "peft>=0.15" "av>=14" "pillow>=10" "pandas>=2" "pyarrow" "huggingface_hub" "python-dotenv" "openai"
!pip uninstall -y -q torchao  # Colab 既定の古い torchao (0.10) を peft が拒否するので外す
!pip install -q -e . --no-deps

## 2. Drive とフレーム

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
!mkdir -p /content/pt_frames {DRIVE}/grpo_4b
!unzip -q -o {DRIVE}/pt_frames_f32_s640.zip -d /content/pt_frames
import os
os.makedirs('/content/pt_videos', exist_ok=True)
for vid in os.listdir('/content/pt_frames'):
    open(f'/content/pt_videos/{vid}.mp4', 'a').close()
print('フレーム', len(os.listdir('/content/pt_frames')), '本')

## 3. 学習前の dev 正解率（貪欲法、300 問。約 10 分）

In [ ]:
%cd /content/movie_llm
!python scripts/pt_difficulty.py --items data/perception_test/dev_grpo.jsonl --out {DRIVE}/grpo_4b/dev_before.jsonl \
    --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --greedy --video-dir /content/pt_videos --frames-dir /content/pt_frames

## 4. GRPO 学習（まず 20 ステップで時間を測る → 300 に伸ばす）

In [ ]:
%cd /content/movie_llm
!python scripts/train_grpo.py --items data/perception_test/train_grpo.jsonl --out {DRIVE}/grpo_4b \
    --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --video-dir /content/pt_videos --frames-dir /content/pt_frames \
    --group-size 8 --prompts-per-step 4 --micro-batch 4 --lr 2e-5 --beta 0.02 \
    --max-steps 20 --save-every 10

In [ ]:
# 学習の記録を表にする
import json, pandas as pd
rows = [json.loads(l) for l in open(f'{DRIVE}/grpo_4b/train_log.jsonl')]
df = pd.DataFrame(rows)
print(df[['step','reward','acc','parsable','kl','loss','len','zero_adv_groups','elapsed']].tail(20).to_string(index=False))
print('1 ステップあたり', round(df.elapsed.diff().median()), '秒')

## 5. 学習後の dev 正解率（最新のチェックポイント）

In [ ]:
%cd /content/movie_llm
latest = open(f'{DRIVE}/grpo_4b/latest.txt').read().strip()
print('adapter:', latest)
!python scripts/pt_difficulty.py --items data/perception_test/dev_grpo.jsonl --out {DRIVE}/grpo_4b/dev_after.jsonl \
    --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --greedy --video-dir /content/pt_videos --frames-dir /content/pt_frames --adapter {latest}
!python scripts/pt_difficulty.py --items data/perception_test/dev_grpo.jsonl --out {DRIVE}/grpo_4b/dev_before.jsonl --summarize
!python scripts/pt_difficulty.py --items data/perception_test/dev_grpo.jsonl --out {DRIVE}/grpo_4b/dev_after.jsonl --summarize

終わったら、手順 4 の表と手順 5 の 2 つの集計を手元の私に貼る。学習後の LoRA（`latest.txt` のフォルダ）はダウンロードして `results/grpo_4b/` に置くと、手元でもベンチマーク評価に使える。